# Getting started — 1BRC

A `SparkSession` is already created for you as `spark` — every new kernel in this JupyterLab instance gets one automatically (see `~/.ipython/profile_default/startup/00-spark.py`).

By default it runs in `local[*]` mode, right here on this node — no cluster required. To attach to the real standalone master/worker cluster instead, set `SPARK_MASTER_URL=spark://$(hostname):7077` before starting Jupyter (or export it and restart the `jupyter` systemd service) and start a new kernel.

`job_01_schema_read.py` → `job_03_format_output.py` in this same folder are the progressive lessons behind the challenge (see `../PROCESS.md`); `job.py` is the capstone that scales to the full 1B rows. `solve_01_10k_rows.py` → `solve_04_1b_rows.py` are that same solve actually run at each scale tier — self-contained, they generate their own tier's real data on first run (from `~/1brc-data-gen`, seeded alongside this notebook) and report wall-clock time. Copy any of their code into cells here to run it interactively.

If `~/data/measurements.txt` doesn't exist yet, generate a small sample first (from your own machine: `make generate-data ROWS=100000 STATIONS=50`, or on the node: `python3 ~/1brc-data-gen/generate_measurements.py --rows 100000 --stations 50 --output ~/data/measurements.txt`).

In [ ]:
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

In [ ]:
import os
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

MEASUREMENT_SCHEMA = StructType([
    StructField("station", StringType(), False),
    StructField("temperature", DoubleType(), False),
])

input_path = os.path.expanduser("~/data/measurements.txt")
df = spark.read.csv(input_path, sep=";", schema=MEASUREMENT_SCHEMA, header=False)
df.printSchema()
print(f"row count: {df.count()}")

In [ ]:
from pyspark.sql import functions as F

agg = df.groupBy("station").agg(
    F.min("temperature").alias("min_temp"),
    F.avg("temperature").alias("mean_temp"),
    F.max("temperature").alias("max_temp"),
).orderBy("station")

agg.explain()

parts = [
    f"{r.station}={r.min_temp:.1f}/{r.mean_temp:.1f}/{r.max_temp:.1f}"
    for r in agg.collect()
]
print("{" + ", ".join(parts) + "}")